# Penguins classification

In this notebook we create a model to classify penguin species. There are 3 species of penguin:

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/01-Fundamentals-of-Machine-Learning/palmer-penguins.png" width=1000>



## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector

from sklearn.linear_model import LogisticRegression

👉 Run the cell below to make sure all our transformers output Pandas DataFrames, and not NumPy arrays.

In [ ]:
from sklearn import set_config
set_config(transform_output="pandas")

## Load the dataset

In [ ]:
penguin_df = sns.load_dataset('penguins')
penguin_df.dropna(inplace=True)

penguin_df.head()

❓What are the characteristics of the penguins?

<img src='https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/gentoo.png' width=700 alt="Image of a penguin highlighting bill and flipper">

❓How many species of penguin are there in this dataset?

In [ ]:
penguin_df['species'].nunique()

❓How many observations for each species are there in the dataset?

In [ ]:
penguin_df['species'].value_counts()

In [ ]:
len(penguin_df)

## Data visualization

In [ ]:
sns.pairplot(penguin_df, hue="species", corner=True);

The **gentoo** species seems to have clearly longer flippers than the other 2 species.

In [ ]:
penguin_df.groupby(["island", "species"]).size() \
          .reset_index() \
          .pivot(columns=['species'], index=['island'], values=0) \
          .fillna(0).astype(int)

Chinstrap and Gentoe only live on one island. Torgersen only has Adelie. All-in-all there is no one-to-one relation though.

In [ ]:
penguin_df.groupby(["sex", "species"]).size() \
          .reset_index() \
          .pivot(columns=['species'], index=['sex'], values=0) \
          .fillna(0).astype(int)

## Train-test-split

In [ ]:
X = penguin_df.drop(columns=['species'])
y = penguin_df['species']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Preprocessing

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20,5))
for i, col in enumerate(['bill_length_mm','bill_depth_mm', 'flipper_length_mm', 'body_mass_g']):
    sns.kdeplot(data=penguin_df, x=col, ax=axes[i])

In [ ]:
preprocessor = ColumnTransformer(
    [
        (
            "num",
            StandardScaler(),
            make_column_selector(dtype_include="number"),
        ),
        (
            "cat",
            OneHotEncoder(drop="if_binary", sparse_output=False),
            make_column_selector(dtype_include="object"),
        ),
    ]
)
preprocessor

## Modeling

In [ ]:
model = LogisticRegression()

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
    ])

cv_score = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')

cv_score.mean()

In [ ]:
final_pipeline = pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)

accuracy_score(y_test, y_pred)

# 🏁 